# Instituições Financeiras — BCB (IFData)

Dados cadastrais das instituições financeiras autorizadas a funcionar pelo Banco Central do Brasil, obtidos via API OData do IFData.

In [ ]:
from bcb.odata.api import IFDATA

ifdata = IFDATA()

# Período de referência: dezembro/2024 (último trimestre disponível)
ANO_MES = 202412

ep = ifdata.get_endpoint("IfDataCadastro")
df = ep.get(AnoMes=ANO_MES)

print(f"Total de instituições: {len(df)}")
df.head()

In [ ]:
# Distribuição por segmento de tamanho (Basileia III)
print("Segmento de tamanho (Basel III):")
print(df["SegmentoTb"].value_counts())

print("\nSituação:")
print(df["Situacao"].value_counts())

print("\nAtividade:")
print(df["Atividade"].value_counts())

In [ ]:
# Exportar para CSV
df.to_csv("instituicoes_financeiras_202412.csv", index=False)
print("Arquivo salvo: instituicoes_financeiras_202412.csv")

## Índice de Basileia por Instituição

O Índice de Basileia (IB) mede a adequação de capital das instituições financeiras: relação entre o Patrimônio de Referência (PR) e os Ativos Ponderados pelo Risco (RWA). O mínimo regulatório no Brasil é 8%, com adicional de conservação de 2,5%.

Fonte: BCB IFData — Relatório 5 (Informações de Capital), referência dez/2024.

In [ ]:
import pandas as pd
from bcb.odata.api import IFDATA

ifdata = IFDATA()
ANO_MES = 202412

# Dados cadastrais para nomes e segmentos
ep_cadastro = ifdata.get_endpoint("IfDataCadastro")
df_cadastro = ep_cadastro.get(AnoMes=ANO_MES)[["CodInst", "NomeInstituicao", "SegmentoTb", "Situacao"]]

# Relatório 5 (Informações de Capital) — nível conglomerado prudencial
ep_valores = ifdata.get_endpoint("IfDataValores")
df_capital = ep_valores.get(AnoMes=ANO_MES, TipoInstituicao=1, Relatorio="5")

# Filtra Índice de Basileia (conta 79664)
df_basileia = df_capital[df_capital["Conta"] == "79664"][["CodInst", "Saldo"]].copy()
df_basileia.rename(columns={"Saldo": "IndiceBasileia"}, inplace=True)

# Join direto: os códigos do relatório batem com CodInst do cadastro
df_merge = df_basileia.merge(df_cadastro, on="CodInst", how="left")
df_merge["IndiceBasileia_%"] = (df_merge["IndiceBasileia"] * 100).round(2)
df_merge = df_merge.sort_values("IndiceBasileia_%", ascending=False).reset_index(drop=True)

print(f"Total de conglomerados prudenciais: {len(df_merge)}")
print(f"Sem nome (não encontrados no cadastro): {df_merge['NomeInstituicao'].isna().sum()}")
df_merge[["NomeInstituicao", "SegmentoTb", "Situacao", "IndiceBasileia_%"]].head(20)

In [ ]:
# Estatísticas resumidas
print("=== Estatísticas do Índice de Basileia (%) — dez/2024 ===")
print(df_merge["IndiceBasileia_pct"].describe().round(2).to_string())
print(f"\nInstituições abaixo do mínimo regulatório (8%): {(df_merge['IndiceBasileia_pct'] < 8).sum()}")
print(f"Instituições entre 8% e 10,5% (zona de atenção): {((df_merge['IndiceBasileia_pct'] >= 8) & (df_merge['IndiceBasileia_pct'] < 10.5)).sum()}")
print(f"Instituições acima de 10,5% (adequadas):          {(df_merge['IndiceBasileia_pct'] >= 10.5).sum()}")

In [ ]:
# Exportar para CSV
df_export = df_merge[["CodInst", "NomeInstituicao", "SegmentoTb", "Situacao", "IndiceBasileia_%"]].copy()
df_export.to_csv("indice_basileia_202412.csv", index=False)
print("Arquivo salvo: indice_basileia_202412.csv")
df_export.head(10)